In [1]:
import os
import glob
import pandas as pd

In [2]:
TRAIN_DIR = r"train_data"

### Находим все parquet-файлы train по маске и сортируем.

In [3]:
train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, "train_data_*.pq")))
print("files:", len(train_files))
print(*train_files[:3], sep="\n")

files: 12
train_data\train_data_0.pq
train_data\train_data_1.pq
train_data\train_data_10.pq


In [4]:
df0 = pd.read_parquet(train_files[0]) 
print(df0.shape)
display(df0.head(3))
print(df0.columns.tolist())
print(df0.dtypes.head(20))

(1974724, 61)


,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,...,enc_paym_21,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag
0,0,1,18,9,2,3,16,10,11,3,...,3,3,3,4,1,3,4,1,0,0
1,0,2,18,9,14,14,12,12,0,3,...,0,0,0,4,1,3,4,1,0,0
2,0,3,18,9,4,8,1,11,11,0,...,0,0,0,4,1,2,3,1,1,1


['id', 'rn', 'pre_since_opened', 'pre_since_confirmed', 'pre_pterm', 'pre_fterm', 'pre_till_pclose', 'pre_till_fclose', 'pre_loans_credit_limit', 'pre_loans_next_pay_summ', 'pre_loans_outstanding', 'pre_loans_total_overdue', 'pre_loans_max_overdue_sum', 'pre_loans_credit_cost_rate', 'pre_loans5', 'pre_loans530', 'pre_loans3060', 'pre_loans6090', 'pre_loans90', 'is_zero_loans5', 'is_zero_loans530', 'is_zero_loans3060', 'is_zero_loans6090', 'is_zero_loans90', 'pre_util', 'pre_over2limit', 'pre_maxover2limit', 'is_zero_util', 'is_zero_over2limit', 'is_zero_maxover2limit', 'enc_paym_0', 'enc_paym_1', 'enc_paym_2', 'enc_paym_3', 'enc_paym_4', 'enc_paym_5', 'enc_paym_6', 'enc_paym_7', 'enc_paym_8', 'enc_paym_9', 'enc_paym_10', 'enc_paym_11', 'enc_paym_12', 'enc_paym_13', 'enc_paym_14', 'enc_paym_15', 'enc_paym_16', 'enc_paym_17', 'enc_paym_18', 'enc_paym_19', 'enc_paym_20', 'enc_paym_21', 'enc_paym_22', 'enc_paym_23', 'enc_paym_24', 'enc_loans_account_holder_type', 'enc_loans_credit_status',

In [5]:
mem_mb = df0.memory_usage(deep=True).sum() / 1024**2
print(f"df0 RAM ~ {mem_mb:.1f} MB")

df0 RAM ~ 919.0 MB


### Читаем данные батчами, чтобы не переполнить память

In [6]:
BATCH_SIZE = 2

n = len(train_files)
i = 0  
batch_files = train_files[i:i+BATCH_SIZE]

df_batch = pd.concat([pd.read_parquet(f) for f in batch_files], ignore_index=True)

bi = i // BATCH_SIZE
print("batch", bi, "shape", df_batch.shape)
df_batch.head(2)


batch 0 shape (4082029, 61)


,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,...,enc_paym_21,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag
0,0,1,18,9,2,3,16,10,11,3,...,3,3,3,4,1,3,4,1,0,0
1,0,2,18,9,14,14,12,12,0,3,...,0,0,0,4,1,3,4,1,0,0


In [7]:
df_batch["id"].nunique()


500000

In [8]:
df_batch.groupby("id").size().describe()


count    500000.000000
mean          8.164058
std           5.859644
min           1.000000
25%           4.000000
50%           7.000000
75%          11.000000
max          51.000000
dtype: float64

### Группируем по id - заявке, считаем базовые признаки

In [9]:
g = df_batch.groupby("id")  

features = g.agg(
    loans_cnt=("rn", "size"),
    rn_max=("rn", "max"),
    rn_min=("rn", "min")
).reset_index()

features.head()


,id,loans_cnt,rn_max,rn_min
0,0,10,10,1
1,1,14,14,1
2,2,3,3,1
3,3,15,15,1
4,4,1,1,1


### Считаем числовые признаки

In [10]:
numeric_cols = df_batch.select_dtypes(include=["number"]).columns
numeric_cols = [c for c in numeric_cols if c not in ["id", "rn"]]

features_numeric = g[numeric_cols].agg(["mean", "min", "max"])
features_numeric.columns = [f"{col}_{stat}" for col, stat in features_numeric.columns]
features_numeric = features_numeric.reset_index()

### Объединяем

In [11]:
features_full = features.merge(features_numeric, on="id", how="left")
features_full.head()

,id,loans_cnt,rn_max,rn_min,pre_since_opened_mean,pre_since_opened_min,pre_since_opened_max,pre_since_confirmed_mean,pre_since_confirmed_min,pre_since_confirmed_max,...,enc_loans_credit_type_max,enc_loans_account_cur_mean,enc_loans_account_cur_min,enc_loans_account_cur_max,pclose_flag_mean,pclose_flag_min,pclose_flag_max,fclose_flag_mean,fclose_flag_min,fclose_flag_max
0,0,10,10,1,8.100000,1,18,7.600000,0,12,...,4,1.0,1,1,0.100000,0,1,0.200000,0,1
1,1,14,14,1,11.428571,2,18,7.642857,3,14,...,4,1.0,1,1,0.071429,0,1,0.142857,0,1
2,2,3,3,1,8.333333,0,13,10.666667,9,14,...,4,1.0,1,1,0.666667,0,1,0.666667,0,1
3,3,15,15,1,7.000000,1,18,7.333333,0,16,...,5,1.0,1,1,0.333333,0,1,0.400000,0,1
4,4,1,1,1,12.000000,12,12,9.000000,9,9,...,3,1.0,1,1,1.000000,1,1,1.000000,1,1


# Итоговый цикл

In [12]:
BATCH_SIZE = 2
parts = []

for i in range(0, len(train_files), BATCH_SIZE):
    batch_files = train_files[i:i+BATCH_SIZE]

    df_batch = pd.concat(
        [pd.read_parquet(f) for f in batch_files],
        ignore_index=True
    )

    g = df_batch.groupby("id")

    
    features_base = g.agg(
        loans_cnt=("rn", "size"),
        rn_max=("rn", "max"),
        rn_min=("rn", "min")
    ).reset_index()

    numeric_cols = df_batch.select_dtypes(include=["number"]).columns
    numeric_cols = [c for c in numeric_cols if c not in ["id", "rn"]]

    features_num = g[numeric_cols].agg(["mean", "min", "max"])
    features_num.columns = [f"{col}_{stat}" for col, stat in features_num.columns]
    features_num = features_num.reset_index()

  
    features_full = features_base.merge(features_num, on="id", how="left")
    parts.append(features_full)

    print("batch", i // BATCH_SIZE, "->", features_full.shape)


all_features = pd.concat(parts, ignore_index=True)
print("all_features:", all_features.shape)


base_agg = {"loans_cnt": "sum", "rn_max": "max", "rn_min": "min"}
other_cols = [c for c in all_features.columns if c not in ["id", "loans_cnt", "rn_max", "rn_min"]]
other_agg = {c: "mean" for c in other_cols}

final_features = all_features.groupby("id", as_index=False).agg({**base_agg, **other_agg})
print("final_features:", final_features.shape)

final_features.head()


batch 0 -> (500000, 181)
batch 1 -> (500000, 181)
batch 2 -> (500000, 181)
batch 3 -> (500000, 181)
batch 4 -> (500000, 181)
batch 5 -> (500000, 181)
all_features: (3000000, 181)
final_features: (3000000, 181)


,id,loans_cnt,rn_max,rn_min,pre_since_opened_mean,pre_since_opened_min,pre_since_opened_max,pre_since_confirmed_mean,pre_since_confirmed_min,pre_since_confirmed_max,...,enc_loans_credit_type_max,enc_loans_account_cur_mean,enc_loans_account_cur_min,enc_loans_account_cur_max,pclose_flag_mean,pclose_flag_min,pclose_flag_max,fclose_flag_mean,fclose_flag_min,fclose_flag_max
0,0,10,10,1,8.100000,1.0,18.0,7.600000,0.0,12.0,...,4.0,1.0,1.0,1.0,0.100000,0.0,1.0,0.200000,0.0,1.0
1,1,14,14,1,11.428571,2.0,18.0,7.642857,3.0,14.0,...,4.0,1.0,1.0,1.0,0.071429,0.0,1.0,0.142857,0.0,1.0
2,2,3,3,1,8.333333,0.0,13.0,10.666667,9.0,14.0,...,4.0,1.0,1.0,1.0,0.666667,0.0,1.0,0.666667,0.0,1.0
3,3,15,15,1,7.000000,1.0,18.0,7.333333,0.0,16.0,...,5.0,1.0,1.0,1.0,0.333333,0.0,1.0,0.400000,0.0,1.0
4,4,1,1,1,12.000000,12.0,12.0,9.000000,9.0,9.0,...,3.0,1.0,1.0,1.0,1.000000,1.0,1.0,1.000000,1.0,1.0


In [13]:
final_features.to_parquet("final_features.parquet", index=False)
